# Demo - Space-filling Shapes in $\mathbb{R}^3$

This is a minimal setup to generate similar space tilings as in the paper (Sec. 4.4).

This notebook requires more dependencies than the `dvx-python` module. Take sure to install the dependencies:

````powershell
python -m venv .env
.env/Scripts/Activate.ps1 # depends on your os
pip install -r demos/requirements.txt
````

In [ ]:
import torch
import tqdm
import dvx.torch as dvx
import arap # as-rigid-as-possible energy in torch
import util # utility functions to keep the notebook clean
import polyscope as ps

torch.manual_seed(0)

## Setup parameters

This optimizes a tiling with one or multiple shapes specified in the `mesh_paths` list. The number of instances for each shape class is specified in the `l` list.

In [ ]:
### configuration
mesh_paths = ["./meshes/spot.obj"]  # path to (multiple) the input meshes
init_scale = 0.5                     # initial scale of the meshes
l = [27]                             # number for each shape shapes
block_res = 64                       # block resolution
step = 0.001                         # learning rate
steps = 1000                         # maximum number of optimization steps
alpha = 0.005                        # weight of arap loss
eps=1e-3                             # converged if max vertex change over all instances is below eps

cs = len(mesh_paths) # number of shape classes
ln = sum(l) # total number of instances

# initialize all instances in for all shape classes
Vs, Vs_init, Fs = [], [], []
for path in mesh_paths: 
  V, F = util.load_obj(path)
  V *= init_scale
  Vs_init.append(V.clone())
  V.requires_grad_(True)
  Fs.append(F)
  Vs.append(V)

### initial transformation
T = torch.rand((ln,3)) * 2/3 - 1/3 
## alternative: uniform grid positions
# spacing = torch.linspace(-1/3,1/3,l) * l/(l+1) 
# T = torch.stack(torch.meshgrid(spacing, spacing, spacing, indexing='ij'), dim=3).flatten(end_dim=2) # grid position
angles = torch.zeros(ln, 3) # we found better minima when not everything is random
# angles = torch.rand(l**3, 3)*2*torch.pi # alternative: (not uniform) random rotation 
T.requires_grad_(True); angles.requires_grad_(True);

## Loss definition

In [ ]:
def loss_fn() -> torch.Tensor:
  # transform shapes
  R = util.rotation3D(angles)
  instances = []
  for c in range(cs):
    inst = torch.bmm(Vs[c].repeat(l[c], 1, 1), R[:l[c]].transpose(1,2)) + T[:l[c]].unsqueeze(1)
    instances.append(inst)

  # deformation loss
  loss_deform = torch.tensor(0.)
  for c in range(cs):
    loss_deform += l[c] * arap.ARAP_energy(Vs[c], Vs_init[c], Fs[c]) # we only need arap for the base shape

  # overlap loss for each instance of each shape class
  box = torch.zeros(block_res, block_res, block_res)
  for c in range(cs):
    for i in range(l[c]):
      q = dvx.voxelize(3*block_res, instances[c][i], Fs[c])
      # everything that exits on one side enters on the other side
      q_blocks = q.unfold(0,block_res,block_res).unfold(1,block_res,block_res).unfold(2,block_res,block_res) # TODO this can be parallelized over instances
      q_repeated = q_blocks.flatten(end_dim=2).sum(0)
      box += q_repeated
  loss_overlap = (box - 1).pow(2).sum()

  loss = 1./(block_res**3) * loss_overlap + alpha * loss_deform
  return loss, instances

## Optimization loop

In [ ]:
init_loss, _ = loss_fn()
print(f"Initial loss: {init_loss:.5f}.")

# optimize
itr = tqdm.trange(steps, desc=f"Loss: -")
optim = torch.optim.Adam([*Vs, T, angles], lr=step)
try:
  for t in itr:
    loss, instances_prev = loss_fn()
    itr.set_description(f"Loss: {loss:.5f}.")
    optim.zero_grad()
    loss.backward()
    optim.step()
    
    # converged?
    R = util.rotation3D(angles.detach())
    max_change = 0
    for c in range(cs):
      instances_new = torch.bmm(Vs[c].detach().repeat(l[c], 1, 1), R[:l[c]].transpose(1,2)) + T[:l[c]].detach().unsqueeze(1)
      change = torch.linalg.vector_norm(instances_prev[c].detach()-instances_new, dim=2).max().item()
      if change > max_change: max_change = change
    if max_change < eps: break
except KeyboardInterrupt:
  print(f"Skipping {steps - t} iterations.")

## Show results

In [ ]:
ps.init()
# register all instances of all classes
for c in range(cs):
  instances = torch.einsum('bij,vj->bvi', R[:l[c]], Vs[c]) + T[:l[c]].unsqueeze(1)
  for i, verts in enumerate(instances):
    ps_mesh = ps.register_surface_mesh(f"instance {c}-{i}", verts.detach().numpy(), Fs[c].detach().numpy())
    ps_mesh.set_transparency(0.5)

# register the tiling cube
s = 1/3 # half-size
cube_v = torch.tensor([
    [-s, -s, -s], [ s, -s, -s], [ s,  s, -s], [-s,  s, -s],  # bottom
    [-s, -s,  s], [ s, -s,  s], [ s,  s,  s], [-s,  s,  s],  # top
])
cube_e = torch.tensor([
    [0,1],[1,2],[2,3],[3,0],  # bottom face
    [4,5],[5,6],[6,7],[7,4],  # top face
    [0,4],[1,5],[2,6],[3,7],  # vertical edges
])
ps_net = ps.register_curve_network("tiling_cube", cube_v.numpy(), cube_e.numpy())
ps_net.set_color((0, 0, 0))
ps_net.set_radius(0.003)

ps.show()